# 🍽️ **Demonstrating Integration of Llumo with LangChain**

---

## 📖 Notebook Overview

This notebook demonstrates how to integrate **Llumo** with **LangChain** for evaluating responses generated by a conversational agent. We use a restaurant ordering system as an example, where a user interacts with the bot to view the menu, add or remove items from the cart, and place an order. After the agent generates responses, **Llumo** is used to evaluate the quality of those outputs.

---


## **Install Necessary Libraries**

In [ ]:
!pip install langchain_community

# **Basic Imports🧪**

In [ ]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.tools import tool
import uuid


 # **Define the Restaurant Menu, Cart, and Order History✌️**

In [ ]:
# -------- Menu and State Setup --------
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

cart = {}
orderHistory = {}

# -------- Tool Definitions --------
tool_outputs = []
@tool
def getMenu() -> str:
    """Get the restaurant menu."""
    return str(menu)

@tool
def addToCart(item: str, quantity: int) -> str:
    """Add an item to the cart."""
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity
            menu[item]["stock"] -= quantity
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    return str({"error": "Item not available in menu."})

@tool
def removeFromCart(item: str, quantity: int) -> str:
    """Remove an item from the cart."""
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity
            menu[item]["stock"] += quantity
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            menu[item]["stock"] += cart[item]
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    return str({"error": "Item not in cart."})

@tool
def getOrderDetails() -> str:
    """Get the order details and generate an order ID."""
    if not cart:
        return str({"message": "Your cart is empty."})
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    order_id = str(uuid.uuid4())[:8]
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()
    return str({"orderId": order_id, "order": orderHistory[order_id]})

@tool
def clearCart() -> str:
    """Clear all items from the cart."""
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

@tool
def viewOrderHistory() -> str:
    """View past order history."""
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})



# **Set Up OpenAI API Key🔐**

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "Your Own OpenAi Api key"


# **🛠️ LangChain Agent Setup**


In [ ]:


# Initialize the OpenAI LLM (GPT-4o)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# List of all restaurant tools available for the agent
tools = [getMenu, addToCart, removeFromCart, getOrderDetails, clearCart, viewOrderHistory]

# Create an agent using OpenAI Functions mode
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True  # Set to True to see internal agent reasoning
)


# **🔄 Run Agent Over Sample Queries and Collect Outputs**


In [ ]:

# List to store results of each interaction
results = []

# Sample user queries to test the agent's functionality
queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
    "Remove 1 burger from my cart",
    "Place my order"
]

# Loop through each query and get agent responses
for query in queries:
    tool_outputs.clear()  # Reset tool outputs for each query
    print("\n🧠 User:", query)

    # Run the agent on the current query
    output = agent.run(query)

    # Format the tool call responses
    tool_output_str = "\n".join([f"{name}: {out}" for name, out in tool_outputs])

    # Append the interaction result to the list
    results.append({
        "user_query": query,
        "message_history": f"User: {query}\nTool Calls:\n{tool_output_str}",
        "output": output,
        "all_tool_descriptions": "\n".join([f"{t.name}: {t.description}" for t in tools])
    })



🧠 User: Show me the menu


> Entering new AgentExecutor chain...

Invoking: `getMenu` with `{}`


{'burger': {'price': 150, 'stock': 10, 'description': 'Delicious beef burger'}, 'pizza': {'price': 300, 'stock': 5, 'description': 'Cheesy pepperoni pizza'}, 'pasta': {'price': 250, 'stock': 8, 'description': 'Creamy alfredo pasta'}, 'coke': {'price': 50, 'stock': 20, 'description': 'Refreshing soft drink'}, 'sandwich': {'price': 120, 'stock': 15, 'description': 'Grilled cheese sandwich'}, 'fries': {'price': 100, 'stock': 12, 'description': 'Crispy golden french fries'}, 'mojito': {'price': 180, 'stock': 10, 'description': 'Cool mint mojito'}, 'coffee': {'price': 120, 'stock': 20, 'description': 'Hot brewed coffee'}, 'tea': {'price': 80, 'stock': 25, 'description': 'Refreshing herbal tea'}}Here's the menu with available items:

1. **Burger**
   - Price: 150
   - Description: Delicious beef burger
   - Stock: 10

2. **Pizza**
   - Price: 300
   - Description: Cheesy pepperoni pizza
   - St

# **📊 Displaying Agent DataFrame**


In [ ]:

import pandas as pd

# Convert the results list into a DataFrame
df = pd.DataFrame(results)

df

,user_query,message_history,output,all_tool_descriptions
0,Show me the menu,User: Show me the menu\nTool Calls:\n,Here's the menu with available items:\n\n1. **...,getMenu: Get the restaurant menu.\naddToCart: ...
1,Add 2 burgers to my cart,User: Add 2 burgers to my cart\nTool Calls:\n,I've added 2 burgers to your cart.,getMenu: Get the restaurant menu.\naddToCart: ...
2,Add 1 coke to my cart,User: Add 1 coke to my cart\nTool Calls:\n,1 coke has been added to your cart. Your curre...,getMenu: Get the restaurant menu.\naddToCart: ...
3,Remove 1 burger from my cart,User: Remove 1 burger from my cart\nTool Calls:\n,I have removed 1 burger from your cart. Your c...,getMenu: Get the restaurant menu.\naddToCart: ...
4,Place my order,User: Place my order\nTool Calls:\n,"To place your order, I need to know what items...",getMenu: Get the restaurant menu.\naddToCart: ...


Saving the dataframe to a csv (optional)

In [ ]:
df.to_csv("response.csv",index=False)

# **🚀Now we will evalaute our Bot with Llumo🚀**

In [ ]:
!pip install llumo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.81.0
    Uninstalling openai-1.81.0:
      Successfully uninstalled openai-1.81.0


## **🔑 Setup OpenAI API Key & Llumo API key from Colab User Data**

In [ ]:
from google.colab import userdata

# Retrieve your OpenAI API key from Colab's stored secrets
# ⚠️ Ensure that the required key are saved in Colab using: userdata.set('key_name_here', 'your-api-key-here')

llumo_key = userdata.get("LLUMO_API_KEY")

# **Importing the dataframe (Optional)**

In [ ]:
df = pd.read_csv("response.csv")

Preparing the data frame in required format

In [ ]:
# 🤖 Prepare DataFrame & Initialize Llumo Client

from llumo import LlumoClient

# Rename required columns to match Llumo input format
df.rename(columns={
    "user_query": "query",
    "messageHistory": "messageHistory",
    "all_tool_descriptions": "tools"
}, inplace=True)


# ✅ Evaluate Agent Responses using Llumo

In [ ]:

client = LlumoClient(api_key=llumo_key)

# Evaluate responses generated by the agent using Llumo's evaluation API
results = client.evaluateAgentResponses(
    dataframe=df,
    prompt_template="give answer for the given {{query}}"
)



======= Running evaluation for: Tool Reliability =======

======= Running evaluation for: Stepwise Progression =======

======= Running evaluation for: Tool Selection Accuracy =======

======= Running evaluation for: Final Task Alignment =======


In [ ]:
results

,query,messageHistory,output,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,Show me the menu,User: Show me the menu\nTool Calls:\n,Here's the menu with available items:\n\n1. **...,getMenu: Get the restaurant menu.\naddToCart: ...,1,No tools were used in the conversation. The m...,1,No tools were called when at least one was cle...,99,"No tools were expected, and no tools were used...",2,"The user requested a menu, but no menu was pro..."
1,Add 2 burgers to my cart,User: Add 2 burgers to my cart\nTool Calls:\n,I've added 2 burgers to your cart.,getMenu: Get the restaurant menu.\naddToCart: ...,1,No tools were used. The metric definition spe...,2,No tools were called when at least one was cle...,2,"No tools were used by the assistant, while the...",1,The tool did not add burgers to the cart. The...
2,Add 1 coke to my cart,User: Add 1 coke to my cart\nTool Calls:\n,1 coke has been added to your cart. Your curre...,getMenu: Get the restaurant menu.\naddToCart: ...,1,No tools were used in the conversation. The m...,2,No tools were called when at least one was cle...,100,The user requested adding a coke to the cart. ...,2,The tool did not add a coke to the cart. The ...
3,Remove 1 burger from my cart,User: Remove 1 burger from my cart\nTool Calls:\n,I have removed 1 burger from your cart. Your c...,getMenu: Get the restaurant menu.\naddToCart: ...,1,No tools were used in the conversation. The m...,2,No tools were called when at least one was cle...,100,The user requested removal of an item from the...,1,The user requested removal of a burger from th...
4,Place my order,User: Place my order\nTool Calls:\n,"To place your order, I need to know what items...",getMenu: Get the restaurant menu.\naddToCart: ...,1,No tools were used in the conversation. The m...,1,No tools were called when at least one was cle...,1,No tools were used by the assistant to fulfill...,1,"The user requested to place an order, but the ..."


# Conclusion
This notebook demonstrates the integration of LangChain with Llumo, where we used LangChain's conversational agent for a restaurant ordering system and then evaluated the agent's performance with Llumo. This combination can be extended to build more sophisticated conversational systems and evaluate their effectiveness in various applications.